# Data Preprocessing and Exploratory Data Analysis

## Set Up and Libraries

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# Switch Directories for Google Colab
if os.getcwd() == '/content':
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir("/content/drive/Othercomputers/My MacBook Pro/CreditCardFraudDetection/notebooks")

# Current working directory
print(os.getcwd())

/Users/nazarii.dubelovskyi/PycharmProjects/CreditCardFraudDetection/notebooks


## Data Preprocessing

### Loading Data

In [3]:
# Read the Data
train_transaction = pd.read_csv("../data/train_transaction.csv")
train_identity = pd.read_csv("../data/train_identity.csv")
train_identity["identity"] = 1

# Merge the training data and Drop unnecessary identification column
df = pd.merge(train_transaction, train_identity, on='TransactionID', how='left').drop(columns=["TransactionID"])
df["identity"] = df["identity"].fillna(0)

In [4]:
# Check the main dataset for balance
# Each column value is either 0 (no fraud, negative) or 1 (fraud, positive)
categories = df["isFraud"]
fraud = categories.sum()
no_fraud = len(categories) - fraud
print(fraud, no_fraud)
print("{:.1f}% of transactions is classified as fraud".format((fraud/len(categories))*100)) # Imbalanced

# Compare whether "identity" features can be omitted in case of too many missing values
df_i = pd.merge(train_transaction, train_identity, on='TransactionID', how='right').drop(columns=["TransactionID"])
categories_i = df_i["isFraud"]
fraud_i = categories_i.sum()
no_fraud_i = len(categories_i) - fraud_i
print(fraud_i, no_fraud_i)
print("{:.1f}% of transactions is classified as fraud".format((fraud_i/len(categories_i))*100))
# The identity dataset, even though smaller, has a twice larger percentage of transactions identified as fraud
# Need to be careful removing the identity-contributed features that might have too many missing values

20663 569877
3.5% of transactions is classified as fraud
11318 132915
7.8% of transactions is classified as fraud


In [9]:
# Split into X and y
X = df.drop(columns="isFraud")
y = df["isFraud"]

# X-y train-test split
# Train: X - features, y - target (trained on)
# Test (evaluation): X - features, y - target (checked against)
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=0)

### Handling Missing Values

In [7]:
# Columns with many missing values are present
print("Percentage of missing values in non-full columns: ")
for v in df.isnull().sum():
    if v > 0:
        print((v/df.shape[0])*100, '% missing values')

Null Values at Each Column: 
1.5126833068039423 % missing values
0.26501168422122123 % missing values
0.267043722694483 % missing values
0.7212043214684865 % missing values
0.26602770345785215 % missing values
11.12642666034477 % missing values
11.12642666034477 % missing values
59.6523520845328 % missing values
93.62837403054831 % missing values
15.99485216920107 % missing values
76.75161716395164 % missing values
0.21488806854743114 % missing values
47.54919226470688 % missing values
44.514850814508755 % missing values
28.60466691502693 % missing values
52.4674027161581 % missing values
87.60676668811597 % missing values
93.40992989467267 % missing values
87.31229044603245 % missing values
87.31229044603245 % missing values
12.873302401192129 % missing values
47.29349409015477 % missing values
89.04104717715988 % missing values
89.50926270870728 % missing values
89.46946862193924 % missing values
15.09008703898127 % missing values
45.90713584177194 % missing values
45.90713584177194 

In [10]:
# Missing Values: Drop at cutoffs first
# For the non-identity categories, apply normal cutoff at ~40%
# Non-identity columns are the first 394 columns of df
ni_cols = X_train.columns[:394]
ni_cols_keep = X_train[ni_cols].loc[:, X_train[ni_cols].isnull().mean() <= 0.4]

# For the identity categories, cutoff at ~80% since identity has high influence
i_cols = X_train.columns[394:]
i_cols_keep = X_train[i_cols].loc[:, X_train[i_cols].isnull().mean() <= 0.8]

# Save the name of the dropped columns
ni_cols_dropped = ni_cols.difference(ni_cols_keep.columns)
i_cols_dropped = i_cols.difference(i_cols_keep.columns)
cutoff_dropped_cols = ni_cols_dropped.union(i_cols_dropped)

# Concatenate the dataframe
X_train = pd.concat([ni_cols_keep, i_cols_keep], axis=1)

## Exploratory Data Analysis